# 🔥 Building an LLM from Scratch with PyTorch

We build a complete GPT-style language model in PyTorch — from tokenisation
to training and text generation. Every component is annotated.

**Architecture:** nano-GPT (character-level, suitable for CPU training)

**Steps:**
1. Dataset preparation and tokenisation
2. Data loader with sliding window
3. Multi-head self-attention module
4. Feed-forward network (FFN)
5. Transformer block (attention + FFN + residuals)
6. Full GPT model
7. Training loop with loss tracking
8. Text generation with temperature sampling
9. Inference utilities

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch                          # core PyTorch tensor library
import torch.nn as nn                 # neural network building blocks
import torch.nn.functional as F       # stateless ops: softmax, cross_entropy
import numpy as np
import matplotlib.pyplot as plt
import math, time

# Use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Dataset & Tokenisation

We use a small text corpus (Shakespeare excerpt) and build a
character-level vocabulary — the simplest possible tokeniser.

In [ ]:
# ── Tiny Shakespeare corpus (excerpt) ────────────────────────────────────────
TEXT = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die—to sleep,
No more; and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to: 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep, perchance to dream—ay, there's the rub,
For in that sleep of death what dreams may come,
When we have shuffled off this mortal coil,
Must give us pause.
All the world's a stage,
And all the men and women merely players;
They have their exits and their entrances,
And one man in his time plays many parts.
""" * 30   # repeat to get enough training data

# ── Character-level vocabulary ────────────────────────────────────────────────
# Build sorted list of unique characters
chars = sorted(set(TEXT))
vocab_size = len(chars)
print(f'Vocabulary size: {vocab_size} characters')
print(f'Text length: {len(TEXT):,} characters')

# Bidirectional lookup tables
char_to_idx = {ch: i for i, ch in enumerate(chars)}  # char → integer
idx_to_char = {i: ch for i, ch in enumerate(chars)}  # integer → char

def encode(text):
    """Convert string to list of token indices."""
    return [char_to_idx[c] for c in text]

def decode(indices):
    """Convert list of token indices back to string."""
    return ''.join(idx_to_char[i] for i in indices)

# Encode entire corpus as a 1-D tensor
data = torch.tensor(encode(TEXT), dtype=torch.long)
print(f'Encoded tensor shape: {data.shape}')

# Train/validation split: 90% train, 10% val
split = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]
print(f'Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}')

## 2. Data Loader — Sliding Window Batches

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
BLOCK_SIZE   = 64    # context window length (max tokens the model sees)
BATCH_SIZE   = 16    # number of sequences per gradient update
D_MODEL      = 128   # embedding dimension
N_HEADS      = 4     # number of attention heads
N_LAYERS     = 3     # number of transformer blocks
FFN_DIM      = 256   # feed-forward hidden dimension (usually 4 × d_model)
DROPOUT      = 0.1   # dropout probability
MAX_ITERS    = 1000  # training steps (increase for better results)
EVAL_EVERY   = 200   # evaluate every N steps
LR           = 3e-4  # AdamW learning rate

def get_batch(split='train', block_size=BLOCK_SIZE, batch_size=BATCH_SIZE):
    """
    Sample a random batch of (input, target) pairs.
    Each input is a sequence of block_size tokens.
    Each target is the same sequence shifted by 1 (next-token prediction).
    """
    # Choose from train or validation data
    data_split = train_data if split == 'train' else val_data

    # Randomly sample starting positions for each sequence in the batch
    # Start positions must leave room for block_size tokens
    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    # Stack sequences into (batch_size, block_size) tensors
    x = torch.stack([data_split[i      : i + block_size    ] for i in ix])
    y = torch.stack([data_split[i + 1  : i + block_size + 1] for i in ix])  # shifted by 1

    return x.to(device), y.to(device)

# Quick sanity check
xb, yb = get_batch('train')
print(f'Input batch shape:  {xb.shape}')   # (16, 64)
print(f'Target batch shape: {yb.shape}')   # (16, 64)
print(f'Sample input:  "{decode(xb[0][:20].tolist())}"')
print(f'Sample target: "{decode(yb[0][:20].tolist())}"')

## 3. Multi-Head Self-Attention Module

In [ ]:
# ── Multi-Head Causal Self-Attention ─────────────────────────────────────────

class MultiHeadAttention(nn.Module):
    """
    Causal (masked) multi-head self-attention.
    Allows each token to attend to all previous tokens (not future ones).
    """

    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads'

        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads   # dimension per head

        # Single projection for Q, K, V (3× to get all at once)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)

        # Output projection: combines all heads back to d_model
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        # Dropout applied to attention weights and output
        self.attn_drop = nn.Dropout(dropout)
        self.out_drop  = nn.Dropout(dropout)

        # Causal mask: upper triangle is 0 (will be set to -inf)
        # register_buffer: not a parameter, but moves to device with .to()
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(block_size, block_size))  # lower triangle = 1
        )

    def forward(self, x):
        B, T, C = x.shape   # batch, sequence length, d_model

        # Project to Q, K, V in one matrix multiply, then split
        qkv = self.qkv_proj(x)                         # (B, T, 3*C)
        Q, K, V = qkv.split(C, dim=-1)                 # each (B, T, C)

        # Reshape for multi-head: (B, n_heads, T, head_dim)
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        # Scaled dot-product attention: QKᵀ / √head_dim
        scale = self.head_dim ** -0.5
        attn = (Q @ K.transpose(-2, -1)) * scale       # (B, heads, T, T)

        # Apply causal mask: set future positions to -inf → 0 after softmax
        attn = attn.masked_fill(self.mask[:T, :T] == 0, float('-inf'))

        # Softmax over the key dimension
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Weighted sum of values
        out = attn @ V                                  # (B, heads, T, head_dim)

        # Concatenate heads: (B, T, C)
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_drop(self.out_proj(out))

print('MultiHeadAttention defined.')

## 4. Feed-Forward Network & Transformer Block

In [ ]:
# ── Feed-Forward Network ──────────────────────────────────────────────────────

class FeedForward(nn.Module):
    """
    Two-layer MLP with GELU activation applied to each token independently.
    The inner dimension is typically 4× d_model (here controlled by ffn_dim).
    """
    def __init__(self, d_model, ffn_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),   # expand dimension
            nn.GELU(),                     # smooth non-linearity (better than ReLU for LLMs)
            nn.Linear(ffn_dim, d_model),   # project back
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# ── Transformer Block ─────────────────────────────────────────────────────────

class TransformerBlock(nn.Module):
    """
    One Transformer block: pre-norm attention + pre-norm FFN.
    Residual connections allow gradients to flow through the full depth.
    """
    def __init__(self, d_model, n_heads, ffn_dim, block_size, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)  # pre-attention normalisation
        self.attn = MultiHeadAttention(d_model, n_heads, block_size, dropout)
        self.ln2  = nn.LayerNorm(d_model)  # pre-FFN normalisation
        self.ffn  = FeedForward(d_model, ffn_dim, dropout)

    def forward(self, x):
        # Attention sub-layer with residual connection
        x = x + self.attn(self.ln1(x))    # 'pre-norm' style (layer norm before sub-layer)
        # FFN sub-layer with residual connection
        x = x + self.ffn(self.ln2(x))
        return x

print('TransformerBlock defined.')

## 5. Full GPT Model

In [ ]:
# ── GPT Language Model ────────────────────────────────────────────────────────

class NanoGPT(nn.Module):
    """
    Minimal GPT-style causal language model.
    Architecture: token embedding + positional embedding → N transformer blocks → LM head.
    """

    def __init__(self, vocab_size, d_model, n_heads, n_layers, ffn_dim,
                 block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        # Token embedding: maps token id → d_model vector
        self.token_emb = nn.Embedding(vocab_size, d_model)

        # Learnable positional embedding: maps position index → d_model vector
        self.pos_emb   = nn.Embedding(block_size, d_model)

        self.drop  = nn.Dropout(dropout)

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ffn_dim, block_size, dropout)
            for _ in range(n_layers)
        ])

        # Final layer normalisation
        self.ln_final = nn.LayerNorm(d_model)

        # Language model head: project d_model → vocab_size (logits)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: share embedding weights with lm_head
        # This halves the parameter count and improves generalisation
        self.lm_head.weight = self.token_emb.weight

        # Initialise weights (GPT-2 style)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """Apply GPT-2-style weight initialisation."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        """
        Forward pass.
        idx     : (B, T) token index tensor
        targets : (B, T) next-token targets for computing loss (optional)
        Returns : logits (B, T, vocab_size) and optional loss scalar
        """
        B, T = idx.shape
        assert T <= self.block_size, f'Sequence length {T} exceeds block size {self.block_size}'

        # Create position indices [0, 1, ..., T-1]
        positions = torch.arange(T, device=idx.device)   # shape (T,)

        # Token + positional embeddings (broadcast over batch dimension)
        x = self.drop(self.token_emb(idx) + self.pos_emb(positions))

        # Pass through each transformer block
        for block in self.blocks:
            x = block(x)

        # Final normalisation
        x = self.ln_final(x)

        # Project to vocabulary logits
        logits = self.lm_head(x)   # (B, T, vocab_size)

        # Compute cross-entropy loss if targets are provided
        if targets is None:
            return logits, None

        # Flatten batch and time dimensions for F.cross_entropy
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=1.0, top_k=None):
        """
        Autoregressive text generation.
        idx         : (1, T) seed context tensor
        max_new_tokens : how many tokens to generate
        temperature : > 1 = creative, < 1 = conservative
        top_k       : if set, restrict sampling to top-k tokens
        """
        for _ in range(max_new_tokens):
            # Crop context to block_size (can't exceed positional embedding range)
            idx_crop = idx[:, -self.block_size:]

            # Forward pass — get logits for the last position
            logits, _ = self(idx_crop)
            logits = logits[:, -1, :]   # only the last-token logits: (1, vocab_size)

            # Apply temperature scaling
            logits = logits / temperature

            # Optional top-k filtering: zero out all but top-k logits
            if top_k is not None:
                topk_vals = logits.topk(top_k, dim=-1).values
                logits = logits.masked_fill(logits < topk_vals[:, -1:], float('-inf'))

            # Sample from the resulting distribution
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)   # (1, 1)

            # Append the newly generated token to the context
            idx = torch.cat([idx, next_idx], dim=1)

        return idx

# Instantiate model and count parameters
model = NanoGPT(
    vocab_size=vocab_size,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    ffn_dim=FFN_DIM,
    block_size=BLOCK_SIZE,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'NanoGPT parameters: {n_params:,} ({n_params/1e6:.2f}M)')

## 6. Training Loop

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────

# AdamW: Adam + weight decay — the standard LLM optimiser
# Weight decay regularises the model to prevent overfitting
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# Learning rate scheduler: linear warmup then cosine decay
def get_lr(step, warmup_steps=100, max_steps=MAX_ITERS):
    """Cosine LR schedule with warmup — standard for LLM training."""
    if step < warmup_steps:
        return LR * step / warmup_steps    # linear warmup
    # Cosine decay from LR to 0
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return LR * 0.5 * (1 + math.cos(math.pi * progress))

train_losses = []
val_losses   = []

@torch.no_grad()
def estimate_loss(eval_iters=50):
    """Average loss over multiple batches for a stable estimate."""
    model.eval()   # disable dropout for evaluation
    losses = {}
    for split in ['train', 'val']:
        split_losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            split_losses.append(loss.item())
        losses[split] = np.mean(split_losses)
    model.train()  # re-enable dropout
    return losses

start_time = time.time()

for step in range(MAX_ITERS + 1):
    # Update learning rate according to schedule
    lr_now = get_lr(step)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now

    # Evaluate periodically
    if step % EVAL_EVERY == 0:
        losses = estimate_loss()
        elapsed = time.time() - start_time
        print(f'step {step:4d} | train loss {losses["train"]:.4f} | '
              f'val loss {losses["val"]:.4f} | lr {lr_now:.2e} | {elapsed:.1f}s')
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])

    if step == MAX_ITERS: break

    # Forward pass on a training batch
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)

    # Backward pass: compute gradients
    optimizer.zero_grad(set_to_none=True)  # clear previous gradients
    loss.backward()                        # backpropagate

    # Gradient clipping: prevents exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()   # update weights

print('Training complete.')

In [ ]:
# ── Plot training curve ────────────────────────────────────────────────────────
steps_logged = list(range(0, MAX_ITERS + 1, EVAL_EVERY))
plt.figure(figsize=(7, 3))
plt.plot(steps_logged, train_losses, label='Train loss', color='#34d399', lw=2)
plt.plot(steps_logged, val_losses,   label='Val loss',   color='#f59e0b', lw=2, linestyle='--')
plt.xlabel('Training step'); plt.ylabel('Cross-entropy loss')
plt.title('NanoGPT Training Curve')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 7. Text Generation (Inference)

In [ ]:
# ── Text Generation ───────────────────────────────────────────────────────────

model.eval()  # disable dropout for generation

def generate_text(seed_text, max_tokens=200, temperature=0.8, top_k=20):
    """
    Generate text continuation given a seed string.
    seed_text   : starting prompt
    max_tokens  : tokens to generate
    temperature : creativity control
    top_k       : restrict sampling to top-k tokens
    """
    # Encode seed text to token indices
    context = torch.tensor(encode(seed_text), dtype=torch.long,
                           device=device).unsqueeze(0)  # (1, T)

    # Generate tokens autoregressively
    with torch.no_grad():
        output_idx = model.generate(
            context, max_new_tokens=max_tokens,
            temperature=temperature, top_k=top_k
        )

    # Decode the full output (seed + generated)
    return decode(output_idx[0].tolist())

# Demo generation at different temperatures
seed = 'To be, or not to'
for temp in [0.5, 1.0, 1.5]:
    print(f'\n--- Temperature {temp} ---')
    print(generate_text(seed, max_tokens=120, temperature=temp, top_k=15))

## Architecture Summary

```
Input tokens (B, T)
       ↓
token_emb + pos_emb → Dropout
       ↓
[TransformerBlock × N_LAYERS]
  ├── LayerNorm
  ├── MultiHeadAttention (causal mask)
  │     ├── QKV projection
  │     ├── Split → N heads
  │     ├── Scaled dot-product attention
  │     └── Concat + output projection
  ├── Residual connection
  ├── LayerNorm
  ├── FeedForward (GELU)
  └── Residual connection
       ↓
Final LayerNorm → LM Head (d_model → vocab_size)
       ↓
Logits (B, T, vocab_size) → Cross-Entropy Loss
```